In [2]:
#import google.generativeai as genai
import json
import os
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

In [ ]:
# --- Configuration (same as before) ---
genai.configure(api_key="ENTER YOUR KEY HERE")
model = genai.GenerativeModel('gemini-2.5-pro')

In [ ]:
df_advice = pd.read_csv('../../data/data_participants.csv')
df_advice = df_advice.query("cond == 'ADV'")
all_texts = df_advice['ADVICE'].tolist()

In [ ]:
def get_overall_score(text: str) -> float | None:
    prompt = f"""

    Imagine you are a research assistant in a cognitive science lab.

    You will read short pieces of advice written by participants about a two-step decision-making task.  

    The task can be summarized as several key components and your task is to identify and evaluate these components in the advice following below criteria:

    - Fixed Pairs
    2 points	Explicitly states that gnomes always appear in fixed pairs (e.g., “There are four pairs of gnomes, and each pair always appears together”).
    1 points	Vague mention of gnomes being paired, without specifying that the pairs are fixed or always the same.
    0 points	No mention
    
    - Gnome-to-Forest Mapping
    2 points	Explicitly states that each gnome color consistently leads to one specific forest/basket (e.g., “The blue gnome always leads to the same forest”).
    1 points	Mentions that gnomes lead to forest/basket, but without clearly specifying which gnome color always leads to the same forest/basket.
    0 points	No mention

    
    - Dynamic Rewards
    2 points	Explicitly states that the number of mushrooms in each forest/basket changes over time (e.g., “Rewards fluctuate, so you need to track them”).
    1 points	Vague mention of mushrooms or rewards without indicating that they change over time.
    0 points	No mention

    
    - Color Belief
    2 points	Explicitly states that gnome color determines the amount of reward (e.g., “Red gnomes give more mushrooms”).
    1 points	Vague suggestion that some gnome colors might be better or worse than others.
    0 points	No mention

    - Irrelevant Features
    2 points	Explicitly states irrelevant features or strategies as meaningful (e.g., “Choose the taller gnome,” “Focus hard and you’ll get more mushrooms”).
    1 points	Vague mention of irrelevant details or strategies without strong emphasis.
    0 points	No mention

    Here is an example of an advice found in the task, the score you should have provided and an explanation for it:
    “green blue orange and yellow are one basket, the other 4 colors the other basket... start with either collection and do it until you notice the numbers decreasing. once the numbers start decreasing, switch to the other color collection as they will begin to increase”
    - Fixed Pairs: 0. Not mentioned.
    - Gnome-to-Forest Mapping: 1. Indicates that transition from gnome to forest are fixed but fails to mention which gnome colors.
    - Dynamic Rewards: 2. Indicates that rewards are moving and instruct how to play around it.
    - Color Belief: 0. Not mentioned.
    - Irrelevant Features: 0. None.

    Your final output must be a single JSON object containing a dictionary including five keys (fixed_pairs, gnome_forest_mapping, dynamic_rewards, color_reward_mapping, irrelevant_features) and two values (score, justifications):

    Here is the Text: "{text}"
    
    """
    try:
        response = model.generate_content(prompt)
        clean_json_str = response.text.strip().replace("```json", "").replace("```", "")
        result = json.loads(clean_json_str)
        return result
    except Exception as e:
        print(f"Could not process text '{text}'. Error: {e}")
        return None


In [ ]:
# --- Loop and Process ---
for call in range(6,10):  # Retry up to 10 times
    annotated_results = []
    for text in all_texts:
        result = get_overall_score(text)
        if result is not None:
            # Initialize default values
            fixed_pair = None
            transition_function = None
            dynamic_rewards = None
            color_reward_mapping = None
            irrelevant_features = None
            
            # Safely extract scores with error handling
            try:
                if 'fixed_pairs' in result:
                    fixed_pair = result['fixed_pairs'].get('score') if isinstance(result['fixed_pairs'], dict) else result['fixed_pairs']
                
                if 'gnome_forest_mapping' in result:
                    transition_function = result['gnome_forest_mapping'].get('score') if isinstance(result['gnome_forest_mapping'], dict) else result['gnome_forest_mapping']
                
                if 'dynamic_rewards' in result:
                    dynamic_rewards = result['dynamic_rewards'].get('score') if isinstance(result['dynamic_rewards'], dict) else result['dynamic_rewards']
                
                if 'color_reward_mapping' in result:
                    color_reward_mapping = result['color_reward_mapping'].get('score') if isinstance(result['color_reward_mapping'], dict) else result['color_reward_mapping']
                
                if 'irrelevant_features' in result:
                    irrelevant_features = result['irrelevant_features'].get('score') if isinstance(result['irrelevant_features'], dict) else result['irrelevant_features']
                
                # Only append if we got at least one valid score
                if any(score is not None for score in [fixed_pair, transition_function, dynamic_rewards, color_reward_mapping, irrelevant_features]):
                    annotated_results.append({
                        'ID': df_advice[df_advice['text'] == text]['ID'].values[0], 
                        'gen': df_advice[df_advice['text'] == text]['gen'].values[0],  
                        "text": text, 
                        "fixed_pair": fixed_pair,
                        "transition_function": transition_function,
                        "color_reward_mapping": color_reward_mapping,
                        "dynamic_rewards": dynamic_rewards,
                        "irrelevant_features": irrelevant_features
                    })
                    
                    print(f"fixed_pair: {fixed_pair}, transition_function: {transition_function}, color_reward_mapping: {color_reward_mapping}, dynamic_rewards: {dynamic_rewards}, irrelevant_features: {irrelevant_features} of Processed text: {text[:50]}...")
                else:
                    print(f"No valid scores extracted for text: {text[:50]}...")
                    
            except Exception as e:
                print(f"Error processing result for text '{text[:50]}...': {e}")
                print(f"Result structure: {result}")
        else:
            print(f"No result returned for text: {text[:50]}...")
            
        time.sleep(1) # Be a good citizen and avoid hitting rate limits

    df_advice_annotated = pd.DataFrame(annotated_results)
    print(f"Call {call+1}: Processed {len(annotated_results)} texts successfully")
    df_advice_annotated.to_csv(f'../../data/raw/LLM/LLM_advice_annotated_call-{call+1}.csv', index=False)

In [ ]:
results_dir = '../../data/raw/LLM/'
file_paths = []
for folder, subs, files in os.walk(results_dir):
  for filename in files:
    if filename.endswith(".csv") and 'score' in filename and 'LLM' in filename:
      file_paths.append(os.path.abspath(os.path.join(folder, filename)))

In [ ]:
df_annotated_results  = [pd.read_csv(f, sep=',') for f in file_paths]
df_annotated_results = pd.concat(df_annotated_results, ignore_index=True)
df_annotated_results = df_annotated_results.to_csv('../../data/data_llm.csv', index=False)

[]